# Семинар 7. Обучение без учителя

В этом семинаре мы разберем:
- Кластеризация (KMeans, DBSCAN, иерархическая кластеризация)
- Снижение размерности (PCA, t-SNE, UMAP)
- Обнаружение аномалий (LOF)
- Приближенный поиск ближайших соседей (HNSW)

In [ ]:
# Colab: install deps; locally use `uv run jupyter lab seminar.ipynb`
import sys
if "google.colab" in sys.modules:
    !pip install -q umap-learn hnswlib networkx

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import (
    make_blobs, make_moons, make_circles, load_iris, load_digits,
)
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import LocalOutlierFactor
from scipy.cluster.hierarchy import dendrogram, linkage
import umap

np.random.seed(42)

## 1. Кластеризация

### 1.1 KMeans

Алгоритм:
1. Случайно инициализируем K центроидов
2. Присваиваем каждую точку ближайшему центроиду
3. Пересчитываем центроиды как среднее точек кластера
4. Повторяем 2-3 до сходимости

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=500, centers=4, cluster_std=1.0, random_state=42)

km = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = km.fit_predict(X_blobs)

plt.figure(figsize=(10, 7))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='tab10', s=20, alpha=0.7)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c='red', marker='X', s=200, edgecolors='k', label='centroids')
plt.title(f'KMeans (K=4, silhouette={silhouette_score(X_blobs, labels):.3f})')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

#### Elbow method и Silhouette score

Как выбрать K? Два подхода:
- **Elbow method**: ищем "локоть" на графике inertia (сумма квадратов расстояний до центроидов)
- **Silhouette score**: мера того, насколько точки похожи на свой кластер по сравнению с соседними

In [ ]:
K_range = range(2, 10)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_blobs)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_blobs, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'o-')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow method')
axes[0].grid(True)

axes[1].plot(K_range, silhouettes, 'o-')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette analysis')
axes[1].grid(True)

plt.tight_layout()
plt.show()

### 1.2 DBSCAN

Density-based clustering: не нужно указывать K. Находит области высокой плотности, разделенные областями низкой плотности. Умеет находить кластеры произвольной формы и отмечать выбросы.

Параметры:
- `eps` - радиус окрестности
- `min_samples` - минимум точек в окрестности для core point

In [ ]:
X_moon, y_moon = make_moons(n_samples=500, noise=0.1, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# KMeans fails on moons
km_labels = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_moon)
axes[0].scatter(X_moon[:, 0], X_moon[:, 1], c=km_labels, cmap='tab10', s=20)
axes[0].set_title('KMeans (K=2) - fails on non-convex')
axes[0].grid(alpha=0.2)

# DBSCAN works
db_labels = DBSCAN(eps=0.2, min_samples=5).fit_predict(X_moon)
axes[1].scatter(X_moon[:, 0], X_moon[:, 1], c=db_labels, cmap='tab10', s=20)
axes[1].set_title(f'DBSCAN (eps=0.2) - handles moons')
axes[1].grid(alpha=0.2)

# DBSCAN with wrong eps
db_labels2 = DBSCAN(eps=0.5, min_samples=5).fit_predict(X_moon)
axes[2].scatter(X_moon[:, 0], X_moon[:, 1], c=db_labels2, cmap='tab10', s=20)
axes[2].set_title(f'DBSCAN (eps=0.5) - too large eps')
axes[2].grid(alpha=0.2)

plt.suptitle('KMeans vs DBSCAN on make_moons', fontsize=14)
plt.tight_layout()
plt.show()

### 1.3 Иерархическая кластеризация

Agglomerative (bottom-up): начинаем с N кластеров (каждая точка - кластер), на каждом шаге объединяем два ближайших. Результат визуализируется как дендрограмма.

In [ ]:
X_hier = X_blobs[:100]  # subset for readable dendrogram

Z = linkage(X_hier, method='ward')

plt.figure(figsize=(16, 6))
dendrogram(Z, truncate_mode='lastp', p=20)
plt.title('Dendrogram (Ward linkage, top 20 merges)')
plt.xlabel('Cluster')
plt.ylabel('Distance')
plt.grid(True, axis='y')
plt.show()

In [ ]:
# Разные методы связи (linkage)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, method in zip(axes, ['ward', 'complete', 'average']):
    agg = AgglomerativeClustering(n_clusters=4, linkage=method)
    labels = agg.fit_predict(X_blobs)
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='tab10', s=20)
    ax.set_title(f'Agglomerative ({method})')
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 1.4 Сравнение методов кластеризации

In [ ]:
# 4 разных типа данных
datasets = [
    ('Blobs', *make_blobs(n_samples=500, centers=3, cluster_std=1.0, random_state=42)),
    ('Moons', *make_moons(n_samples=500, noise=0.1, random_state=42)),
    ('Circles', *make_circles(n_samples=500, noise=0.05, factor=0.5, random_state=42)),
    ('Anisotropic', *(lambda: (
        np.dot(make_blobs(n_samples=500, centers=3, cluster_std=0.8, random_state=42)[0],
               [[0.6, -0.6], [-0.4, 0.8]]),
        make_blobs(n_samples=500, centers=3, cluster_std=0.8, random_state=42)[1]
    ))()),
]

clusterers = [
    ('KMeans', lambda X: KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X)),
    ('DBSCAN', lambda X: DBSCAN(eps=0.5, min_samples=5).fit_predict(X)),
    ('Agglomerative', lambda X: AgglomerativeClustering(n_clusters=3).fit_predict(X)),
]

fig, axes = plt.subplots(len(datasets), len(clusterers), figsize=(18, 20))

for i, (dname, X, y_true) in enumerate(datasets):
    X_scaled = StandardScaler().fit_transform(X)
    for j, (cname, cluster_fn) in enumerate(clusterers):
        labels = cluster_fn(X_scaled)
        axes[i][j].scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, cmap='tab10', s=10)
        axes[i][j].set_title(f'{cname}' if i == 0 else '')
        if j == 0:
            axes[i][j].set_ylabel(dname, fontsize=14)
        axes[i][j].grid(alpha=0.2)

plt.suptitle('Comparison of clustering methods', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 2. Снижение размерности

### 2.1 PCA (Principal Component Analysis)

Находит направления максимальной дисперсии (главные компоненты) и проецирует данные на них. Линейный метод, быстрый, интерпретируемый.

In [ ]:
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)
y_iris = iris.target

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_iris)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for cls in np.unique(y_iris):
    mask = y_iris == cls
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=iris.target_names[cls], s=30)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[0].set_title('PCA: Iris (4D -> 2D)')
axes[0].legend()
axes[0].grid(alpha=0.2)

# Scree plot
pca_full = PCA().fit(X_iris)
axes[1].bar(range(1, 5), pca_full.explained_variance_ratio_, alpha=0.7, label='Individual')
axes[1].plot(range(1, 5), np.cumsum(pca_full.explained_variance_ratio_), 'ro-', label='Cumulative')
axes[1].set_xlabel('Component')
axes[1].set_ylabel('Explained variance ratio')
axes[1].set_title('Scree plot')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### 2.2 t-SNE

Нелинейный метод, сохраняет локальную структуру: близкие точки остаются близкими. Хорош для визуализации высокоразмерных данных.

Параметр `perplexity` ~ количество ближайших соседей, которые учитываются.

In [ ]:
digits = load_digits()
X_digits = StandardScaler().fit_transform(digits.data)
y_digits = digits.target
print(f'Digits: {X_digits.shape[0]} samples, {X_digits.shape[1]} features, {len(np.unique(y_digits))} classes')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, perp in zip(axes, [5, 30, 100]):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42)
    X_tsne = tsne.fit_transform(X_digits)
    scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.7)
    ax.set_title(f't-SNE (perplexity={perp})')
    ax.grid(alpha=0.2)

plt.colorbar(scatter, ax=axes, shrink=0.6, label='Digit')
plt.suptitle('t-SNE on digits dataset (64D -> 2D)', fontsize=14)
plt.tight_layout()
plt.show()

**Внимание:** расстояния между кластерами в t-SNE НЕ информативны. Только расстояния внутри кластеров имеют смысл.

### 2.3 UMAP

Быстрая альтернатива t-SNE, лучше сохраняет глобальную структуру. Ключевые параметры:
- `n_neighbors` - размер локальной окрестности (~perplexity в t-SNE)
- `min_dist` - минимальное расстояние между точками в проекции

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, (n_neighbors, min_dist) in zip(axes, [(5, 0.1), (15, 0.1), (50, 0.5)]):
    reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, random_state=42)
    X_umap = reducer.fit_transform(X_digits)
    scatter = ax.scatter(X_umap[:, 0], X_umap[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.7)
    ax.set_title(f'UMAP (n_neighbors={n_neighbors}, min_dist={min_dist})')
    ax.grid(alpha=0.2)

plt.colorbar(scatter, ax=axes, shrink=0.6, label='Digit')
plt.suptitle('UMAP on digits dataset (64D -> 2D)', fontsize=14)
plt.tight_layout()
plt.show()

### 2.4 Сравнение: PCA vs t-SNE vs UMAP

In [ ]:
X_pca_d = PCA(n_components=2).fit_transform(X_digits)
X_tsne_d = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_digits)
X_umap_d = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_digits)

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
for ax, X_proj, title in zip(axes,
    [X_pca_d, X_tsne_d, X_umap_d],
    ['PCA', 't-SNE', 'UMAP'],
):
    scatter = ax.scatter(X_proj[:, 0], X_proj[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.7)
    ax.set_title(title, fontsize=14)
    ax.grid(alpha=0.2)

plt.colorbar(scatter, ax=axes, shrink=0.6, label='Digit')
plt.suptitle('Dimensionality reduction: digits (64D -> 2D)', fontsize=14)
plt.tight_layout()
plt.show()

| Метод | Линейный? | Скорость | Глобальная структура | Локальная структура | Использование |
|---|---|---|---|---|---|
| PCA | Да | Быстрый | Сохраняет | Средне | Preprocessing, шумоподавление |
| t-SNE | Нет | Медленный | Не сохраняет | Отлично | Визуализация |
| UMAP | Нет | Быстрый | Сохраняет лучше t-SNE | Отлично | Визуализация + downstream задачи |

## 3. Обнаружение аномалий

### Local Outlier Factor (LOF)

LOF оценивает "локальную плотность" каждой точки по сравнению с ее соседями. Точки с существенно меньшей плотностью - аномалии. В отличие от Isolation Forest (sem4), LOF учитывает локальный контекст.

In [ ]:
# Данные с выбросами
X_normal, _ = make_blobs(n_samples=300, centers=2, cluster_std=0.5, random_state=42)
rng = np.random.RandomState(42)
X_outliers = rng.uniform(low=-5, high=8, size=(20, 2))
X_lof = np.vstack([X_normal, X_outliers])

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.06)
lof_labels = lof.fit_predict(X_lof)
lof_scores = -lof.negative_outlier_factor_

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Predictions
axes[0].scatter(X_lof[:, 0], X_lof[:, 1], c=lof_labels, cmap='coolwarm', s=30)
axes[0].set_title('LOF predictions (red=outlier)')
axes[0].grid(alpha=0.2)

# Anomaly scores
scatter = axes[1].scatter(X_lof[:, 0], X_lof[:, 1], c=lof_scores, cmap='Reds', s=30)
plt.colorbar(scatter, ax=axes[1], label='LOF score')
axes[1].set_title('LOF anomaly scores (higher = more anomalous)')
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

n_outliers_found = (lof_labels == -1).sum()
print(f'Найдено аномалий: {n_outliers_found} (добавлено: {len(X_outliers)})')

## 4. Приближенный поиск ближайших соседей

### Зачем нужен приближенный поиск?

Точный поиск K ближайших соседей (brute-force) имеет сложность $O(n \cdot d)$ на один запрос. При миллионах векторов и сотнях измерений это слишком медленно. HNSW (Hierarchical Navigable Small World) - один из самых эффективных алгоритмов приближенного поиска (ANN - Approximate Nearest Neighbors).

### Как работает HNSW

Идея: строим многоуровневый граф поверх точек данных.

- **Уровень 0** (нижний): содержит все точки, каждая связана с M ближайшими соседями
- **Уровень 1**: содержит подмножество точек (примерно 1/M от уровня 0)
- **Уровень 2**: еще меньше точек
- ...и так далее

Поиск: начинаем с верхнего уровня (мало точек, грубая навигация), спускаемся вниз (больше точек, точная навигация). На каждом уровне жадно перемещаемся к ближайшему соседу запроса.

Ключевые параметры:
- **M** - количество связей на узел (больше M = точнее, но больше памяти)
- **ef_construction** - ширина поиска при построении графа
- **ef** - ширина поиска при запросе (больше ef = точнее, но медленнее)

In [ ]:
import hnswlib
import networkx as nx
import time as _time
from sklearn.neighbors import NearestNeighbors

### 4.1 HNSW: построение индекса и поиск

In [ ]:
# Генерируем 2D данные для наглядности
np.random.seed(42)
n_points = 200
X_hnsw = np.random.randn(n_points, 2).astype(np.float32)

# Строим HNSW индекс
dim = 2
index = hnswlib.Index(space='l2', dim=dim)
index.init_index(max_elements=n_points, ef_construction=50, M=8)
index.add_items(X_hnsw, np.arange(n_points))
index.set_ef(30)

# Запрос: ищем 5 ближайших соседей для случайной точки
query = np.array([[1.5, 1.0]], dtype=np.float32)
labels_hnsw, distances_hnsw = index.knn_query(query, k=5)

print(f"Query point: {query[0]}")
print(f"Nearest neighbors (indices): {labels_hnsw[0]}")
print(f"Distances (L2 squared): {distances_hnsw[0]}")

# Сравним с точным поиском
nn_exact = NearestNeighbors(n_neighbors=5, metric='euclidean').fit(X_hnsw)
dist_exact, idx_exact = nn_exact.kneighbors(query)
print(f"\nExact neighbors (indices):   {idx_exact[0]}")
print(f"Recall@5: {len(set(labels_hnsw[0]) & set(idx_exact[0])) / 5:.0%}")

### 4.2 Визуализация графа HNSW

Построим граф соседей, чтобы увидеть структуру HNSW: как точки связаны между собой.

In [ ]:
# Строим граф HNSW: для каждой точки находим M ближайших соседей из индекса
M_vis = 6
labels_all, _ = index.knn_query(X_hnsw, k=M_vis + 1)  # +1 т.к. сама точка тоже возвращается

G = nx.Graph()
for i in range(n_points):
    G.add_node(i, pos=(X_hnsw[i, 0], X_hnsw[i, 1]))
    for j in labels_all[i]:
        if j != i:
            G.add_edge(i, int(j))

pos = {i: (X_hnsw[i, 0], X_hnsw[i, 1]) for i in range(n_points)}

plt.figure(figsize=(14, 10))
nx.draw_networkx_edges(G, pos, alpha=0.08, edge_color='steelblue', width=0.5)
nx.draw_networkx_nodes(G, pos, node_size=20, node_color='steelblue', alpha=0.8)

# Подсветим query и его соседей
plt.scatter(*query[0], c='red', s=200, marker='*', zorder=5, label='Query')
for idx in labels_hnsw[0]:
    plt.scatter(X_hnsw[idx, 0], X_hnsw[idx, 1], c='red', s=80, edgecolors='k', zorder=5)
    plt.plot([query[0, 0], X_hnsw[idx, 0]], [query[0, 1], X_hnsw[idx, 1]],
             'r-', linewidth=2, alpha=0.6)

plt.title(f'HNSW graph ({n_points} points, M={M_vis}): edges = neighbor connections', fontsize=14)
plt.legend(fontsize=12)
plt.grid(alpha=0.2)
plt.axis('equal')
plt.show()

### 4.3 Визуализация пути поиска

Симулируем поиск: начинаем с случайной точки, на каждом шаге переходим к ближайшему соседу запроса из текущих связей в графе.

In [ ]:
# Симуляция greedy search по графу
def greedy_search(G, X, query_point, start_node, max_steps=50):
    """Greedy walk: always move to the neighbor closest to query."""
    path = [start_node]
    current = start_node
    for _ in range(max_steps):
        neighbors = list(G.neighbors(current))
        if not neighbors:
            break
        dists = [np.linalg.norm(X[n] - query_point) for n in neighbors]
        best = neighbors[np.argmin(dists)]
        if np.linalg.norm(X[best] - query_point) >= np.linalg.norm(X[current] - query_point):
            break  # local minimum reached
        current = best
        path.append(current)
    return path

# Начнем поиск с далекой точки
start = np.argmax(np.linalg.norm(X_hnsw - query[0], axis=1))
search_path = greedy_search(G, X_hnsw, query[0], start)

plt.figure(figsize=(14, 10))
nx.draw_networkx_edges(G, pos, alpha=0.05, edge_color='gray', width=0.5)
nx.draw_networkx_nodes(G, pos, node_size=15, node_color='lightgray', alpha=0.6)

# Подсветим путь
path_x = [X_hnsw[i, 0] for i in search_path]
path_y = [X_hnsw[i, 1] for i in search_path]
plt.plot(path_x, path_y, 'o-', color='darkorange', linewidth=2.5, markersize=8,
         label=f'Search path ({len(search_path)} steps)', zorder=4)

# Start и end
plt.scatter(X_hnsw[start, 0], X_hnsw[start, 1], c='green', s=150, marker='s',
            edgecolors='k', zorder=5, label='Start (farthest point)')
plt.scatter(*query[0], c='red', s=250, marker='*', zorder=5, label='Query')
plt.scatter(X_hnsw[search_path[-1], 0], X_hnsw[search_path[-1], 1],
            c='red', s=120, edgecolors='k', zorder=5, label='Found nearest')

plt.title(f'Greedy search on HNSW graph: {len(search_path)} hops instead of checking all {n_points} points',
          fontsize=13)
plt.legend(fontsize=11)
plt.grid(alpha=0.2)
plt.axis('equal')
plt.show()

print(f"Шагов поиска: {len(search_path)} (из {n_points} точек)")
print(f"Найденная точка: {search_path[-1]}, расстояние: {np.linalg.norm(X_hnsw[search_path[-1]] - query[0]):.4f}")
print(f"Точный ближайший: {idx_exact[0][0]}, расстояние: {dist_exact[0][0]:.4f}")

In [ ]:
# Бенчмарк на более крупных данных
n_bench = 5000
dim_bench = 50
k = 10

np.random.seed(42)
X_bench = np.random.randn(n_bench, dim_bench).astype(np.float32)
X_queries = np.random.randn(100, dim_bench).astype(np.float32)

# Exact nearest neighbors (ground truth)
nn = NearestNeighbors(n_neighbors=k, metric='euclidean').fit(X_bench)
_, idx_true = nn.kneighbors(X_queries)

# Build HNSW index
idx_hnsw = hnswlib.Index(space='l2', dim=dim_bench)
idx_hnsw.init_index(max_elements=n_bench, ef_construction=200, M=16)
idx_hnsw.add_items(X_bench, np.arange(n_bench))

ef_values = [1, 5, 10, 20, 50, 100, 200, 500]
recalls = []
times = []

for ef in ef_values:
    idx_hnsw.set_ef(ef)
    t0 = _time.time()
    labels_bench, _ = idx_hnsw.knn_query(X_queries, k=k)
    query_time = _time.time() - t0
    
    # Compute recall@k
    recall = np.mean([
        len(set(labels_bench[i]) & set(idx_true[i])) / k
        for i in range(len(X_queries))
    ])
    recalls.append(recall)
    times.append(query_time)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(ef_values, recalls, 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('ef (search width)', fontsize=12)
axes[0].set_ylabel(f'Recall@{k}', fontsize=12)
axes[0].set_title('Recall vs ef')
axes[0].set_xscale('log')
axes[0].grid(True)
axes[0].axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='Perfect recall')
axes[0].legend()

axes[1].plot([r for r in recalls], [t * 1000 for t in times], 'o-', linewidth=2, markersize=8)
for i, ef in enumerate(ef_values):
    axes[1].annotate(f'ef={ef}', (recalls[i], times[i] * 1000), fontsize=9,
                     textcoords="offset points", xytext=(5, 5))
axes[1].set_xlabel(f'Recall@{k}', fontsize=12)
axes[1].set_ylabel('Query time (ms)', fontsize=12)
axes[1].set_title('Speed-Recall tradeoff')
axes[1].grid(True)

plt.suptitle(f'HNSW: {n_bench} vectors, {dim_bench}D, {len(X_queries)} queries', fontsize=14)
plt.tight_layout()
plt.show()

### 4.5 FAISS: индексы от Facebook

FAISS (Facebook AI Similarity Search) - библиотека для эффективного поиска похожих векторов. Поддерживает несколько типов индексов:

- **IndexFlatL2** - точный brute-force поиск (baseline)
- **IndexIVFFlat** - кластеризация + поиск в ближайших кластерах (Inverted File Index)
- **IndexIVFPQ** - IVF + Product Quantization (сжатие векторов)
- **IndexHNSWFlat** - HNSW внутри FAISS

FAISS оптимизирован для GPU и больших датасетов (миллионы/миллиарды векторов).

In [ ]:
import faiss

In [ ]:
# Те же данные, что и для HNSW бенчмарка
# X_bench: (5000, 50), X_queries: (100, 50), idx_true: exact neighbors

# 1. Flat (exact) - baseline
index_flat = faiss.IndexFlatL2(dim_bench)
index_flat.add(X_bench)
D_flat, I_flat = index_flat.search(X_queries, k)

recall_flat = np.mean([
    len(set(I_flat[i]) & set(idx_true[i])) / k
    for i in range(len(X_queries))
])
print(f'Flat (exact):  recall@{k} = {recall_flat:.3f}')

In [ ]:
# 2. IVF (Inverted File Index) - кластеризует данные, ищет в nprobe ближайших кластерах
nlist = 50  # количество кластеров (ячеек Вороного)
quantizer = faiss.IndexFlatL2(dim_bench)
index_ivf = faiss.IndexIVFFlat(quantizer, dim_bench, nlist)
index_ivf.train(X_bench)
index_ivf.add(X_bench)

nprobe_values = [1, 2, 5, 10, 20, 50]
recalls_ivf = []
times_ivf = []

for nprobe in nprobe_values:
    index_ivf.nprobe = nprobe
    t0 = _time.time()
    D_ivf, I_ivf = index_ivf.search(X_queries, k)
    t_ivf = _time.time() - t0
    recall = np.mean([
        len(set(I_ivf[i]) & set(idx_true[i])) / k
        for i in range(len(X_queries))
    ])
    recalls_ivf.append(recall)
    times_ivf.append(t_ivf)
    print(f'IVF nprobe={nprobe:2d}: recall@{k} = {recall:.3f}, time = {t_ivf*1000:.1f}ms')

### 4.6 Визуализация: IVF ячейки Вороного

IVF разбивает пространство на кластеры (ячейки Вороного). При поиске проверяем только `nprobe` ближайших к запросу кластеров.

In [ ]:
# Визуализация на 2D данных
np.random.seed(42)
X_2d = np.random.randn(500, 2).astype(np.float32)
query_2d = np.array([[2.0, 1.5]], dtype=np.float32)

nlist_2d = 10
quantizer_2d = faiss.IndexFlatL2(2)
index_2d = faiss.IndexIVFFlat(quantizer_2d, 2, nlist_2d)
index_2d.train(X_2d)
index_2d.add(X_2d)

# Получаем центроиды кластеров
centroids = quantizer_2d.reconstruct_n(0, nlist_2d)

# Назначаем каждую точку кластеру
_, assignments = quantizer_2d.search(X_2d, 1)
assignments = assignments.ravel()

# Какие кластеры будут проверены при nprobe=3?
_, query_clusters = quantizer_2d.search(query_2d, 3)
probed = set(query_clusters.ravel())

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: все кластеры
for c in range(nlist_2d):
    mask = assignments == c
    color = 'salmon' if c in probed else 'lightgray'
    alpha = 0.8 if c in probed else 0.3
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, s=15, alpha=alpha)
axes[0].scatter(centroids[:, 0], centroids[:, 1], c='blue', marker='+', s=200, linewidths=2, label='centroids')
axes[0].scatter(*query_2d[0], c='red', s=200, marker='*', zorder=5, label='query')
axes[0].set_title(f'IVF: {nlist_2d} clusters, nprobe=3 (red = searched)', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.2)
axes[0].set_aspect('equal')

# Right: Voronoi diagram
from scipy.spatial import Voronoi, voronoi_plot_2d
vor = Voronoi(centroids)
voronoi_plot_2d(vor, ax=axes[1], show_vertices=False, line_colors='steelblue', line_alpha=0.6)
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c='lightgray', s=10, alpha=0.5)
axes[1].scatter(centroids[:, 0], centroids[:, 1], c='blue', marker='+', s=200, linewidths=2)
axes[1].scatter(*query_2d[0], c='red', s=200, marker='*', zorder=5)
# Highlight probed cells
for c in probed:
    mask = assignments == c
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], c='salmon', s=15, alpha=0.8)
axes[1].set_title('Voronoi cells (IVF partitioning)', fontsize=13)
axes[1].set_xlim(X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5)
axes[1].set_ylim(X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5)
axes[1].grid(alpha=0.2)
axes[1].set_aspect('equal')

plt.suptitle('FAISS IVF: search only in nearest Voronoi cells', fontsize=14)
plt.tight_layout()
plt.show()

### 4.7 HNSW vs FAISS IVF: speed-recall tradeoff

In [ ]:
# Сравнение HNSW и FAISS IVF на одних данных
plt.figure(figsize=(10, 6))
plt.plot(recalls, [t * 1000 for t in times], 'o-', linewidth=2, markersize=8, label='HNSW (hnswlib)')
plt.plot(recalls_ivf, [t * 1000 for t in times_ivf], 's-', linewidth=2, markersize=8, label='IVF (FAISS)')

for i, ef in enumerate(ef_values):
    plt.annotate(f'ef={ef}', (recalls[i], times[i] * 1000), fontsize=8,
                 textcoords='offset points', xytext=(5, 5))
for i, nprobe in enumerate(nprobe_values):
    plt.annotate(f'nprobe={nprobe}', (recalls_ivf[i], times_ivf[i] * 1000), fontsize=8,
                 textcoords='offset points', xytext=(5, -10))

plt.xlabel(f'Recall@{k}', fontsize=12)
plt.ylabel('Query time (ms)', fontsize=12)
plt.title(f'HNSW vs FAISS IVF ({n_bench} vectors, {dim_bench}D)', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()

## Итоги

### Кластеризация
- **KMeans**: быстрый, но только выпуклые кластеры, нужно задавать K
- **DBSCAN**: произвольная форма кластеров, не нужно задавать K, чувствителен к eps
- **Hierarchical**: дает дендрограмму, можно выбрать K постфактум

### Снижение размерности
- **PCA**: быстрый, линейный, для preprocessing и визуализации
- **t-SNE**: нелинейный, только для визуализации, медленный
- **UMAP**: нелинейный, быстрее t-SNE, можно использовать для downstream задач

### Аномалии
- **Isolation Forest** (sem4): глобальная аномальность
- **LOF**: локальная аномальность (учитывает плотность соседей)

### Приближенный поиск соседей
- **HNSW**: многоуровневый граф, O(log n) на запрос. Трейдофф recall vs speed через параметр ef
- **FAISS IVF**: кластеризация пространства (ячейки Вороного), поиск в nprobe ближайших кластерах. Масштабируется на миллиарды векторов